# Seminar 9. LLMs Reduction Methods

Presentation: https://docs.google.com/presentation/d/107B2VX4wIXjk1-CbdU9bmJfu9VRWjWdXLwU-Qj8Z7eg/edit?usp=sharing

## Part 1. Distillation

In [1]:
!pip install accelerate -q

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_distilled = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    dtype=torch.float16, 
    device_map="auto"
)

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [24]:
messages = [
    {"role": "user", "content": "Tell me about yourself."},
    {'role': 'assistant', 'content': '<think>\n'}
]

In [25]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model_distilled.device)
generated_ids = model_distilled.generate(
    **model_inputs,
    max_new_tokens=1024
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.




</think>

Greetings! I'm DeepSeek-R1, an artificial intelligence assistant created by DeepSeek. I'm at your service and would be delighted to assist you with any inquiries or tasks you may have.


In [62]:
messages = [
    {"role": "user", "content": "Why are г so sus?"},
    {'role': 'assistant', 'content': '<think>\n'}
]

In [63]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model_distilled.device)
generated_ids = model_distilled.generate(
    **model_inputs,
    max_new_tokens=1024
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<think>
Alright, the user is asking why "г" is so sus. I know that "sus" often means something is suspicious or questionable. So, maybe they're noticing that the letter "г" is appearing a lot or in an unusual way.

I should consider different contexts where "г" might be sus. In Russian, "г" is the letter that starts "господь," which means "lord." Could they be talking about that? Or maybe it's in a different language.

Another angle is that "г" is the fourth letter in the alphabet, so if it's appearing frequently, it might seem suspicious. Maybe they're seeing it in unexpected places.

I should also think about possible typos or mix-ups. Maybe they meant a different letter or word that looks similar to "г."

Additionally, "sus" can relate to technology, like a GPU being sus if it's not performing well. Could they be referring to something like that?

I need to make sure I cover all possibilities without assuming too much. I'll ask them to clarify their question so I can provide a more 

## Part 2. Pruning

For the next two parts we will be using the MMLU dataset to measure performance of models. We will also compare their initial and final sizes.

In [1]:
!pip install -q -U accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 10.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 41.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import torch
from datasets import load_dataset
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from tqdm import tqdm

In [3]:
MODEL_ID = 'Qwen/Qwen3-8B'
SUBSET_RATIO = 0.2

In [4]:
def load_mmlu(ratio=0.2):
    print(f"Loading MMLU subset ({ratio*100}%)...")
    dataset = load_dataset("cais/mmlu", "all", split="test")
    df = dataset.to_pandas()
    
    subset_df = df.groupby('subject', group_keys=False).apply(
        lambda x: x.sample(frac=ratio, random_state=42)
    )
    
    return subset_df

In [5]:
def evaluate_model(model, tokenizer, data_df):
    model.eval()
    choices = ['A', 'B', 'C', 'D']

    subjects = data_df['subject'].unique()
    subject_stats = {sub: {'correct': 0, 'total': 0} for sub in subjects}
    
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
    prompt_template = "Question: {question}\nChoices:\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer:"

    with torch.no_grad():
        for _, row in tqdm(data_df.iterrows(), total=len(data_df), desc="Evaluating"):
            prompt = prompt_template.format(
                question=row['question'],
                a=row['choices'][0], b=row['choices'][1],
                c=row['choices'][2], d=row['choices'][3]
            )
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, choice_ids]
            prediction = torch.argmax(last_logits).item()
            
            is_correct = (prediction == row['answer'])
            subject_stats[row['subject']]['total'] += 1
            if is_correct:
                subject_stats[row['subject']]['correct'] += 1

    results = []
    total_correct = 0
    total_samples = 0

    for sub, stats in subject_stats.items():
        acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
        results.append({'subject': sub, 'accuracy': acc, 'count': stats['total']})
        total_correct += stats['correct']
        total_samples += stats['total']
        
    df_results = pd.DataFrame(results)
    mean_accuracy = total_correct / total_samples
                
    return mean_accuracy, df_results

In [6]:
def get_size_mb(path):
    """Count size of all parameters in Megabytes."""
    total_size = 0
    weight_extensions = ('.safetensors', '.bin', '.pt', '.hqq_pack')
    
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024 * 1024)
    
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            if f.endswith(weight_extensions):
                fp = os.path.join(dirpath, f)
                total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

def calculate_metrics(orig_metric, comp_metric, orig_path, comp_path, orig_size=None):
    if orig_size is None:
        orig_size = get_size_mb(orig_path)
    comp_size = get_size_mb(comp_path)
    
    compression_ratio = orig_size / comp_size
    
    performance_drop = max(0, (orig_metric - comp_metric) / orig_metric)
    
    return {
        "Original Size (MB)": orig_size,
        "Compressed Size (MB)": comp_size,
        "Ratio": compression_ratio,
        "Drop": performance_drop
    }

### Base Model

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_baseline = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    dtype=torch.float16, 
    device_map="auto"
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [9]:
orig_params = sum(p.numel() for p in model_baseline.parameters())
print(f"Original parameters: {orig_params / 1e9:.2f}B")

Original parameters: 8.19B


In [13]:
model_baseline.save_pretrained('qwen3-baseline')
tokenizer.save_pretrained('qwen3-baseline')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen3-baseline/tokenizer_config.json',
 'qwen3-baseline/chat_template.jinja',
 'qwen3-baseline/tokenizer.json')

In [14]:
get_size_mb('qwen3-baseline')

15622.631843566895

In [15]:
test_df = load_mmlu(SUBSET_RATIO)
test_df

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(


,question,subject,choices,answer
83,Statement 1 | If a group has an element of ord...,abstract_algebra,"[True, True, False, False, True, False, False,...",2
53,"Statement 1 | If G, H and K are groups of orde...",abstract_algebra,"[True, True, False, False, True, False, False,...",2
70,"(Z,*) is a group with a*b = a+b+1 for all a, b...",abstract_algebra,"[0, -2, a-2, (2+a)*-1]",3
45,"Statement 1 | For any two groups G and G', the...",abstract_algebra,"[True, True, False, False, True, False, False,...",2
44,"Let A and B be sets, f: A -> B and g: B -> A b...",abstract_algebra,"[True, True, False, False, True, False, False,...",2
...,...,...,...,...
13984,"According to the Korean foundation myth, who ...",world_religions,"[Hwanin, Hwanung, Joseon, Yi]",1
13902,"After the Bar Kochba revolt, where were the t...",world_religions,"[Palestine and Babylonia, Babylonia and Europe...",0
13961,What does the Tripitaka mean?,world_religions,"[Three gems, Three baskets, Three bodhisattvas...",1
14003,When was the State of Israel established?,world_religions,"[1947, 1948, 1945, 1949]",1


In [19]:
orig_metric, detailed_metric = evaluate_model(model_baseline, tokenizer, test_df)
print(f"Original Metric (MMLU subset): {orig_metric:.4f}")

Evaluating: 100%|██████████| 2809/2809 [07:20<00:00,  6.38it/s]

Original Metric (MMLU subset): 0.7106


In [20]:
detailed_metric

,subject,accuracy,count
0,abstract_algebra,0.500000,20
1,anatomy,0.629630,27
2,astronomy,0.966667,30
3,business_ethics,0.800000,20
4,clinical_knowledge,0.830189,53
5,college_biology,0.862069,29
6,college_chemistry,0.700000,20
7,college_computer_science,0.750000,20
8,college_mathematics,0.600000,20
9,college_medicine,0.800000,35


### Pruning

In [7]:
import copy

def prune_model_layers(model, layers_to_remove):
    """
    Drop layers from model.
    layers_to_remove: list of layers indices to remove (e.g. [15, 16, 17, 18])
    """
    # In Llama/Qwen models the layers are stored in model.model.layers
    old_layers = model.model.layers
    new_layers = torch.nn.ModuleList()
    
    for i, layer in enumerate(old_layers):
        if i not in layers_to_remove:
            new_layers.append(layer)
    
    # Replace model layers
    model.model.layers = new_layers
    # Update config file to fix new layers number
    model.config.num_hidden_layers = len(new_layers)
    
    print(f"Layers removed: {layers_to_remove}")
    print(f"New layer count: {len(model.model.layers)}")
    return model

In [8]:
# 1. Load base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

# 2. Drop layers (Qwen3 has 36 layers)
model_pruned = prune_model_layers(model, layers_to_remove=[3, 5, 7, 9])

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Layers removed: [3, 5, 7, 9]
New layer count: 32


In [9]:
pruned_params = sum(p.numel() for p in model_pruned.parameters())
print(f"Pruned parameters: {pruned_params / 1e9:.2f}B")

Pruned parameters: 7.42B


In [10]:
model_pruned.save_pretrained('qwen3-pruned')
tokenizer.save_pretrained('qwen3-pruned')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen3-pruned/tokenizer_config.json',
 'qwen3-pruned/chat_template.jinja',
 'qwen3-pruned/tokenizer.json')

In [12]:
test_df = load_mmlu(SUBSET_RATIO)

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(


In [13]:
pruned_metric, detailed_metric = evaluate_model(model_pruned, tokenizer, test_df)
print(f"Pruned Metric (MMLU subset): {pruned_metric:.4f}")

Evaluating: 100%|██████████| 2809/2809 [06:40<00:00,  7.02it/s]

Pruned Metric (MMLU subset): 0.2816


In [14]:
detailed_metric

,subject,accuracy,count
0,abstract_algebra,0.350000,20
1,anatomy,0.148148,27
2,astronomy,0.366667,30
3,business_ethics,0.200000,20
4,clinical_knowledge,0.207547,53
5,college_biology,0.344828,29
6,college_chemistry,0.350000,20
7,college_computer_science,0.350000,20
8,college_mathematics,0.350000,20
9,college_medicine,0.285714,35


In [15]:
results = calculate_metrics(
    orig_metric=0.7106, 
    comp_metric=pruned_metric, 
    orig_path=None, 
    comp_path='/kaggle/working/qwen3-pruned', 
    orig_size=15622.631843566895)

print("===PRUNING RESULTS===")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

===PRUNING RESULTS===
Original Size (MB): 15622.6318
Compressed Size (MB): 14150.5625
Ratio: 1.1040
Drop: 0.6037


## Part 3. Quantization

In [1]:
!pip install -q -U bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 38.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


### 3.1 Simple on the fly quantization

In [2]:
import torch
from datasets import load_dataset
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os
from tqdm import tqdm

In [3]:
MODEL_ID = 'Qwen/Qwen3-8B'
SUBSET_RATIO = 0.2

In [4]:
def load_mmlu(ratio=0.2):
    print(f"Loading MMLU subset ({ratio*100}%)...")
    dataset = load_dataset("cais/mmlu", "all", split="test")
    df = dataset.to_pandas()
    
    subset_df = df.groupby('subject', group_keys=False).apply(
        lambda x: x.sample(frac=ratio, random_state=42)
    )
    
    return subset_df

In [5]:
def evaluate_model(model, tokenizer, data_df):
    model.eval()
    choices = ['A', 'B', 'C', 'D']

    subjects = data_df['subject'].unique()
    subject_stats = {sub: {'correct': 0, 'total': 0} for sub in subjects}
    
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
    prompt_template = "Question: {question}\nChoices:\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer:"

    with torch.no_grad():
        for _, row in tqdm(data_df.iterrows(), total=len(data_df), desc="Evaluating"):
            prompt = prompt_template.format(
                question=row['question'],
                a=row['choices'][0], b=row['choices'][1],
                c=row['choices'][2], d=row['choices'][3]
            )
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, choice_ids]
            prediction = torch.argmax(last_logits).item()
            
            is_correct = (prediction == row['answer'])
            subject_stats[row['subject']]['total'] += 1
            if is_correct:
                subject_stats[row['subject']]['correct'] += 1

    results = []
    total_correct = 0
    total_samples = 0

    for sub, stats in subject_stats.items():
        acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
        results.append({'subject': sub, 'accuracy': acc, 'count': stats['total']})
        total_correct += stats['correct']
        total_samples += stats['total']
        
    df_results = pd.DataFrame(results)
    mean_accuracy = total_correct / total_samples
                
    return mean_accuracy, df_results

In [6]:
def get_size_mb(path):
    """Count size of all parameters in Megabytes."""
    total_size = 0
    weight_extensions = ('.safetensors', '.bin', '.pt', '.hqq_pack')
    
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024 * 1024)
    
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            if f.endswith(weight_extensions):
                fp = os.path.join(dirpath, f)
                total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

def calculate_metrics(orig_metric, comp_metric, orig_path, comp_path, orig_size=None):
    if orig_size is None:
        orig_size = get_size_mb(orig_path)
    comp_size = get_size_mb(comp_path)
    
    compression_ratio = orig_size / comp_size
    
    performance_drop = max(0, (orig_metric - comp_metric) / orig_metric)
    
    return {
        "Original Size (MB)": orig_size,
        "Compressed Size (MB)": comp_size,
        "Ratio": compression_ratio,
        "Drop": performance_drop
    }

#### 8 bit quantization

In [7]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_quantized = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quantization_config
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [14]:
quantized_params = sum(p.numel() for p in model_quantized.parameters())
print(f"Quantized parameters: {quantized_params / 1e9:.2f}B")

Quantized parameters: 8.19B


In [9]:
model_quantized.save_pretrained('qwen3-quantized')
tokenizer.save_pretrained('qwen3-quantized')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen3-quantized/tokenizer_config.json',
 'qwen3-quantized/chat_template.jinja',
 'qwen3-quantized/tokenizer.json')

In [10]:
test_df = load_mmlu(SUBSET_RATIO)

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(


In [11]:
quantized_metric, detailed_metric = evaluate_model(model_quantized, tokenizer, test_df)
print(f"8bit Quantization Metric (MMLU subset): {quantized_metric:.4f}")

Evaluating:   0%|          | 0/2809 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Evaluating: 100%|██████████| 2809/2809 [18:20<00:00,  2.55it/s]

8bit Quantization Metric (MMLU subset): 0.7095


In [12]:
detailed_metric

,subject,accuracy,count
0,abstract_algebra,0.550000,20
1,anatomy,0.740741,27
2,astronomy,0.933333,30
3,business_ethics,0.800000,20
4,clinical_knowledge,0.849057,53
5,college_biology,0.896552,29
6,college_chemistry,0.700000,20
7,college_computer_science,0.800000,20
8,college_mathematics,0.600000,20
9,college_medicine,0.771429,35


In [13]:
results = calculate_metrics(
    orig_metric=0.7106, 
    comp_metric=quantized_metric, 
    orig_path=None, 
    comp_path='/kaggle/working/qwen3-quantized', 
    orig_size=15622.631843566895)

print("===QUANTIZATION 8BIT RESULTS===")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

===QUANTIZATION 8BIT RESULTS===
Original Size (MB): 15622.6318
Compressed Size (MB): 9004.0266
Ratio: 1.7351
Drop: 0.0015


#### 4 bit quantization

In [7]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_quantized = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quantization_config
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [8]:
quantized_params = sum(p.numel() for p in model_quantized.parameters())
print(f"Quantized parameters: {quantized_params / 1e9:.2f}B")

Quantized parameters: 4.72B


In [9]:
model_quantized.save_pretrained('qwen3-quantized')
tokenizer.save_pretrained('qwen3-quantized')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('qwen3-quantized/tokenizer_config.json',
 'qwen3-quantized/chat_template.jinja',
 'qwen3-quantized/tokenizer.json')

In [10]:
test_df = load_mmlu(SUBSET_RATIO)

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(


In [11]:
quantized_metric, detailed_metric = evaluate_model(model_quantized, tokenizer, test_df)
print(f"4bit Quantization Metric (MMLU subset): {quantized_metric:.4f}")

Evaluating: 100%|██████████| 2809/2809 [50:31<00:00,  1.08s/it]  

4bit Quantization Metric (MMLU subset): 0.6757


In [12]:
detailed_metric

,subject,accuracy,count
0,abstract_algebra,0.500000,20
1,anatomy,0.629630,27
2,astronomy,0.900000,30
3,business_ethics,0.700000,20
4,clinical_knowledge,0.811321,53
5,college_biology,0.827586,29
6,college_chemistry,0.650000,20
7,college_computer_science,0.800000,20
8,college_mathematics,0.400000,20
9,college_medicine,0.800000,35


In [13]:
results = calculate_metrics(
    orig_metric=0.7106, 
    comp_metric=quantized_metric, 
    orig_path=None, 
    comp_path='/kaggle/working/qwen3-quantized', 
    orig_size=15622.631843566895)

print("===QUANTIZATION 4BIT RESULTS===")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

===QUANTIZATION 4BIT RESULTS===
Original Size (MB): 15622.6318
Compressed Size (MB): 6100.7553
Ratio: 2.5608
Drop: 0.0491


### 3.2 Quantization first, then inference

In [1]:
!pip install -q -U accelerate datasets hqq transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 12.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 33.3 MB/s eta 0:00:0000:0100:01m
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import torch
import numpy as np
import random
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from hqq.models.hf.base import AutoHQQHFModel
from hqq.core.quantize import BaseQuantizeConfig

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [4]:
MODEL_ID = 'Qwen/Qwen3-8B'
SUBSET_RATIO = 0.2
# quantization hyperparams
NBITS = 4
GROUP_SIZE = 128

SAVE_PATH = f"qwen3-8b-hqq-{NBITS}bit"
ORIG_SIZE = 15622.631477355957
ORIG_METRIC = 0.7106

In [5]:
def load_mmlu(ratio=0.2):
    print(f"Loading MMLU subset ({ratio*100}%)...")
    dataset = load_dataset("cais/mmlu", "all", split="test")
    df = dataset.to_pandas()
    
    subset_df = df.groupby('subject', group_keys=False).apply(
        lambda x: x.sample(frac=ratio, random_state=42)
    )
    
    return subset_df

In [6]:
def evaluate_model(model, tokenizer, data_df):
    model.eval()
    choices = ['A', 'B', 'C', 'D']

    subjects = data_df['subject'].unique()
    subject_stats = {sub: {'correct': 0, 'total': 0} for sub in subjects}
    
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
    prompt_template = "Question: {question}\nChoices:\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer:"

    with torch.no_grad():
        for _, row in tqdm(data_df.iterrows(), total=len(data_df), desc="Evaluating"):
            prompt = prompt_template.format(
                question=row['question'],
                a=row['choices'][0], b=row['choices'][1],
                c=row['choices'][2], d=row['choices'][3]
            )
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, choice_ids]
            prediction = torch.argmax(last_logits).item()
            
            is_correct = (prediction == row['answer'])
            subject_stats[row['subject']]['total'] += 1
            if is_correct:
                subject_stats[row['subject']]['correct'] += 1

    results = []
    total_correct = 0
    total_samples = 0

    for sub, stats in subject_stats.items():
        acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
        results.append({'subject': sub, 'accuracy': acc, 'count': stats['total']})
        total_correct += stats['correct']
        total_samples += stats['total']
        
    df_results = pd.DataFrame(results)
    mean_accuracy = total_correct / total_samples
                
    return mean_accuracy, df_results

In [7]:
def get_size_mb(path):
    """Count size of all parameters in Megabytes."""
    total_size = 0
    weight_extensions = ('.safetensors', '.bin', '.pt', '.hqq_pack')
    
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024 * 1024)
    
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            if f.endswith(weight_extensions):
                fp = os.path.join(dirpath, f)
                total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)

def calculate_metrics(orig_metric, comp_metric, orig_path, comp_path, orig_size=None):
    if orig_size is None:
        orig_size = get_size_mb(orig_path)
    comp_size = get_size_mb(comp_path)
    
    compression_ratio = orig_size / comp_size
    
    performance_drop = max(0, (orig_metric - comp_metric) / orig_metric)
    
    return {
        "Original Size (MB)": orig_size,
        "Compressed Size (MB)": comp_size,
        "Ratio": compression_ratio,
        "Drop": performance_drop
    }

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

hqq_config = BaseQuantizeConfig(nbits=NBITS, group_size=GROUP_SIZE)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.float16
)

print(f"Quantizing model to {NBITS}-bit...")
AutoHQQHFModel.quantize_model(model, quant_config=hqq_config, compute_dtype=torch.float16, device="cuda:1")

# model.save_pretrained(SAVE_PATH)
AutoHQQHFModel.save_quantized(model, SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model successfully compressed and saved to {SAVE_PATH}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Quantizing model to 4-bit...


100%|██████████| 253/253 [00:40<00:00,  6.24it/s]


Model successfully compressed and saved to qwen3-8b-hqq-4bit


In [10]:
model_hqq_4bit = AutoHQQHFModel.from_quantized(SAVE_PATH)
tokenizer_hqq_4bit = AutoTokenizer.from_pretrained(SAVE_PATH)

100%|██████████| 253/253 [00:00<00:00, 7835.83it/s]


In [11]:
hqq_4bit_params = sum(p.numel() for p in model_hqq_4bit.parameters())
print(f"Quantized parameters: {hqq_4bit_params / 1e9:.2f}B")

Quantized parameters: 4.72B


In [12]:
test_df = load_mmlu(SUBSET_RATIO)

Loading MMLU subset (20.0%)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

/tmp/ipykernel_55/1419417978.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = df.groupby('subject', group_keys=False).apply(


In [13]:
hqq_4bit_metric, hqq_4bit_detailed_metric = evaluate_model(model_hqq_4bit, tokenizer_hqq_4bit, test_df)
print(f"HQQ Quantized Metric (MMLU subset): {hqq_4bit_metric:.4f}")

Evaluating: 100%|██████████| 2809/2809 [35:33<00:00,  1.32it/s]

HQQ Quantized Metric (MMLU subset): 0.6913


In [14]:
hqq_4bit_detailed_metric

,subject,accuracy,count
0,abstract_algebra,0.500000,20
1,anatomy,0.592593,27
2,astronomy,0.966667,30
3,business_ethics,0.800000,20
4,clinical_knowledge,0.867925,53
5,college_biology,0.896552,29
6,college_chemistry,0.650000,20
7,college_computer_science,0.700000,20
8,college_mathematics,0.450000,20
9,college_medicine,0.742857,35


In [16]:
results = calculate_metrics(ORIG_METRIC, hqq_4bit_metric, None, SAVE_PATH, ORIG_SIZE)

print(f"===HQQ {NBITS}BIT COMPRESSION RESULTS===")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

===HQQ 4BIT COMPRESSION RESULTS===
Original Size (MB): 15622.6315
Compressed Size (MB): 5893.8723
Ratio: 2.6507
Drop: 0.0271


Other quantization methods: https://huggingface.co/docs/transformers/quantization/overview